# Secom feature engineering - first

[Stage 0] 데이터 로딩 (secom_base, 244 피처)  
[Stage 1] 피처 엔지니어링 → 2개 데이터셋 추가 생성  
[Stage 2] 모델 실험 (4 데이터셋 × 4 모델 × 2 불균형처리)  
[Stage 3] 결과 비교 → 최적 피처셋 × 모델 × 불균형처리 조합 식별  
[Stage 4] 최적 조합으로 하이퍼파라미터 튜닝  
[Stage 5] Test 셋 최종 평가 (1회)  

In [1]:
import sys
print(sys.executable)

c:\coding\data_mining\.venv\Scripts\python.exe


In [3]:
import pandas as pd
df = pd.read_csv("secom_preprocessed.csv")

X_train = df[df["split"] == "train"].drop(columns=["Pass/Fail", "split"])
X_test  = df[df["split"] == "test"].drop(columns=["Pass/Fail", "split"])
y_train = df[df["split"] == "train"]["Pass/Fail"]
y_test  = df[df["split"] == "test"]["Pass/Fail"]

In [4]:
# -1(정상) → 0, 1(불량) → 1 로 변환 for xgboost
y_train = (y_train == 1).astype(int)
y_test  = (y_test == 1).astype(int)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Train 불량 비율: {y_train.mean():.3f}")
print(f"Test  불량 비율: {y_test.mean():.3f}")

X_train: (1253, 244), X_test: (314, 244)
Train 불량 비율: 0.066
Test  불량 비율: 0.067


### Stage 1-1: FE1 (IF_Anomaly_Score)

In [3]:
from sklearn.ensemble import IsolationForest

# Data Leakage 방지: Train 데이터로만 fit
iso_forest = IsolationForest(n_estimators=100, random_state=42)
iso_forest.fit(X_train)

X_train_fe1 = X_train.copy()
X_test_fe1  = X_test.copy()

# decision_function: 값이 클수록 정상, 작을수록 이상치
# 부호 반전(-1 곱): "값이 클수록 불량 의심"으로 방향 통일
X_train_fe1["IF_Anomaly_Score"] = -iso_forest.decision_function(X_train)
X_test_fe1["IF_Anomaly_Score"]  = -iso_forest.decision_function(X_test)

### Stage 1-2: FE2 (Targeted Interaction Features)

In [4]:
from scipy.stats import pointbiserialr
import numpy as np
from itertools import combinations

# Point-Biserial 상관계수: 연속형 피처 vs 이진 타겟 간 상관관계 측정
# Data Leakage 방지: Train 기준으로만 상위 피처 선정
correlations = {}
for col in X_train.columns:
    corr, _ = pointbiserialr(y_train, X_train[col])
    correlations[col] = abs(corr)

# 타겟과 가장 연관성 높은 상위 5개 피처 선정
top5 = sorted(correlations, key=correlations.get, reverse=True)[:5]

X_train_fe2 = X_train.copy()
X_test_fe2  = X_test.copy()

# C(5,2) = 10개 조합 생성
# Test에는 Train에서 선정한 top5 기준 그대로 적용 (Leakage 방지)
for col_a, col_b in combinations(top5, 2):
    new_col = f"interact_{col_a}_{col_b}"
    X_train_fe2[new_col] = X_train[col_a] * X_train[col_b]
    X_test_fe2[new_col]  = X_test[col_a]  * X_test[col_b]

### Stage 2 : Modeling

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

In [6]:
# CV 및 평가 지표 설정
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

scoring = {
    "recall": "recall",
    "pr_auc": "average_precision"
}

In [7]:
# Flow A용: SMOTEENN이 불균형 처리 담당 → 모델은 기본 파라미터
models_flow_a = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "RandomForest":       RandomForestClassifier(random_state=42),
    "XGBoost":            XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM":           LGBMClassifier(random_state=42, verbose=-1),
}

# Flow B용: 모델 내부 가중치로 불균형 처리
models_flow_b = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "RandomForest":       RandomForestClassifier(random_state=42, class_weight="balanced"),
    "XGBoost":            XGBClassifier(random_state=42, eval_metric="logloss", scale_pos_weight=14.1),
    "LightGBM":           LGBMClassifier(random_state=42, verbose=-1, scale_pos_weight=14.1),
}

In [8]:
# dataset definition
datasets = {
    "secom_base": (X_train,     y_train),
    "secom_fe1":  (X_train_fe1, y_train),
    "secom_fe2":  (X_train_fe2, y_train),
}

In [9]:
# enjoy

results = []

for ds_name, (X, y) in datasets.items():
    for model_name, model in models_flow_a.items():

        # Flow A: SMOTEENN → 모델
        pipe_a = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=42)),
            ("model",    model)
        ])
        scores_a = cross_validate(pipe_a, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({
            "dataset":    ds_name,
            "model":      model_name,
            "imbalance":  "SMOTEENN",
            "recall":     scores_a["test_recall"].mean(),
            "pr_auc":     scores_a["test_pr_auc"].mean(),
            "recall_std": scores_a["test_recall"].std(),
            "pr_auc_std": scores_a["test_pr_auc"].std(),
        })

        # Flow B: 가중치만 적용 (SMOTEENN 없음)
        pipe_b = ImbPipeline([
            ("model", models_flow_b[model_name])
        ])
        scores_b = cross_validate(pipe_b, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        results.append({
            "dataset":    ds_name,
            "model":      model_name,
            "imbalance":  "CostSensitive",
            "recall":     scores_b["test_recall"].mean(),
            "pr_auc":     scores_b["test_pr_auc"].mean(),
            "recall_std": scores_b["test_recall"].std(),
            "pr_auc_std": scores_b["test_pr_auc"].std(),
        })

In [10]:
results_df = pd.DataFrame(results).sort_values("recall", ascending=False).reset_index(drop=True)
print(results_df.to_string())

       dataset               model      imbalance    recall    pr_auc  recall_std  pr_auc_std
0    secom_fe2  LogisticRegression       SMOTEENN  0.522304  0.138201    0.141898    0.035850
1   secom_base  LogisticRegression       SMOTEENN  0.522059  0.146656    0.131287    0.050227
2    secom_fe1  LogisticRegression       SMOTEENN  0.522059  0.147005    0.131287    0.051039
3    secom_fe1             XGBoost       SMOTEENN  0.386275  0.157199    0.113772    0.036530
4    secom_fe2             XGBoost       SMOTEENN  0.369608  0.170985    0.111835    0.054096
5   secom_base             XGBoost       SMOTEENN  0.348775  0.150751    0.105550    0.041649
6    secom_fe1            LightGBM       SMOTEENN  0.341422  0.161386    0.098548    0.055890
7    secom_fe1  LogisticRegression  CostSensitive  0.282108  0.158318    0.110412    0.060411
8    secom_fe2            LightGBM       SMOTEENN  0.281618  0.164220    0.078379    0.046847
9   secom_base  LogisticRegression  CostSensitive  0.274265 

### Stage 3 : evaluation

1. LogReg + SMOTEENN 3개 데이터셋이 Top 3 독식 (recall ~0.52) — 다만 PR-AUC는 0.138~0.147로 낮음  
2. XGBoost + SMOTEENN (recall ~0.35~0.39, PR-AUC ~0.15~0.17) — Recall은 낮지만 PR-AUC는 더 균형 잡힘  
3. CostSensitive 계열 대부분 하위권 — 특히 트리 모델 (RF/XGB/LGBM) CostSensitive는 Top 10에 거의 안 보임 → threshold=0.5 문제 가능성 (Stage 3에서 검증할 부분)  
4. FE1/FE2 효과 미미 — base 대비 의미 있는 개선 없음. 피처 엔지니어링이 큰 도움은 안 됨

### Stage 3~4 : before hyperparameter tuning

1. 24개 조합 전부 predict_proba()로 확률값 추출 
2. Threshold 그리드 (0.05 ~ 0.50, step 0.05) 탐색 
3. 최적 threshold 기준으로 Recall / PR-AUC 재계산 
4. 결과 정리 → Stage 4 진행할 조합 선별

In [11]:
def find_best_threshold(y_true, proba, thresholds=np.arange(0.05, 0.51, 0.05)):
    """
    Recall이 최대인 threshold 반환
    """
    from sklearn.metrics import precision_score, recall_score

    best_threshold = None
    best_recall    = -1
    best_precision = -1

    for t in thresholds:
        y_pred = (proba >= t).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall    = recall_score(y_true, y_pred, zero_division=0)

        if recall > best_recall:
            best_recall    = recall
            best_threshold = t
            best_precision = precision

    return best_threshold, best_recall, best_precision

In [12]:
from sklearn.metrics import average_precision_score

def evaluate_with_threshold(pipeline, X, y, cv):
    recalls    = []
    pr_aucs    = []
    precisions = []
    thresholds = []

    for train_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        pipeline.fit(X_tr, y_tr)
        proba = pipeline.predict_proba(X_val)[:, 1]

        best_t, best_recall, best_precision = find_best_threshold(y_val, proba)

        if best_t is None:
            recalls.append(0.0)
            precisions.append(0.0)
            thresholds.append(None)
        else:
            recalls.append(best_recall)
            precisions.append(best_precision)
            thresholds.append(best_t)

        pr_aucs.append(average_precision_score(y_val, proba))

    return {
        "recall":          np.mean(recalls),
        "recall_std":      np.std(recalls),
        "pr_auc":          np.mean(pr_aucs),
        "pr_auc_std":      np.std(pr_aucs),
        "precision":       np.mean(precisions),
        "best_threshold":  np.mean([t for t in thresholds if t is not None])
    }

In [13]:
#  24개 조합 전부 재실험
results_stage3 = []

for ds_name, (X, y) in datasets.items():
    for model_name, model in models_flow_a.items():

        # Flow A: SMOTEENN
        pipe_a = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=42)),
            ("model",    model)
        ])
        scores_a = evaluate_with_threshold(pipe_a, X, y, cv)
        results_stage3.append({
            "dataset":   ds_name,
            "model":     model_name,
            "imbalance": "SMOTEENN",
            **scores_a
        })

        # Flow B: CostSensitive
        pipe_b = ImbPipeline([
            ("model", models_flow_b[model_name])
        ])
        scores_b = evaluate_with_threshold(pipe_b, X, y, cv)
        results_stage3.append({
            "dataset":   ds_name,
            "model":     model_name,
            "imbalance": "CostSensitive",
            **scores_b
        })

c:\coding\data_mining\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\coding\data_mining\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please

In [14]:
results_stage3_df = pd.DataFrame(results_stage3).sort_values("recall", ascending=False).reset_index(drop=True)
print(results_stage3_df.to_string())

       dataset               model      imbalance    recall  recall_std    pr_auc  pr_auc_std  precision  best_threshold
0   secom_base        RandomForest       SMOTEENN  1.000000    0.000000  0.132403    0.021958   0.066344            0.05
1    secom_fe1        RandomForest       SMOTEENN  1.000000    0.000000  0.135227    0.031975   0.066362            0.05
2    secom_fe2        RandomForest       SMOTEENN  1.000000    0.000000  0.151086    0.040561   0.066466            0.05
3    secom_fe1        RandomForest  CostSensitive  0.839706    0.063562  0.149319    0.052926   0.093284            0.05
4   secom_base        RandomForest  CostSensitive  0.808578    0.093216  0.173453    0.067046   0.090723            0.05
5    secom_fe1             XGBoost       SMOTEENN  0.776225    0.113411  0.157199    0.036530   0.100273            0.05
6    secom_fe2        RandomForest  CostSensitive  0.771814    0.089789  0.152026    0.040361   0.089587            0.05
7   secom_base             XGBoo

이 feature engineering 채택 x